### Install Required Libraries

We need `transformers` for loading the model and tokenizer, and `accelerate` for efficient device mapping (handling GPU/CPU allocation).

In [1]:
!git clone https://huggingface.co/TheFireHacker/bubblspace-moe0.9b-a0.2b
%cd bubblspace-moe0.9b-a0.2b
!pip install -q -r requirements.txt

Cloning into 'bubblspace-moe0.9b-a0.2b'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 64 (delta 21), reused 0 (delta 0), pack-reused 4 (from 1)
Receiving objects: 100% (64/64), 658.20 KiB | 3.66 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/bubblspace-moe0.9b-a0.2b


### Load Model and Tokenizer

Next, we'll initialize the pipeline with the specified model `TheFireHacker/bubblspace-moe0.9b-a0.2b`. Since this is a Mixture-of-Experts (MoE) model, loading it with `device_map="auto"` is recommended to automatically utilize available GPU resources.

In [2]:
!apt-get install -y git-lfs -q
!git lfs pull

import torch
from safetensors.torch import load_file
from transformers import AutoTokenizer
from src.config import load_config
from src.model.moe import probe_grouped_mm
from src.model.transformer import Transformer

cfg = load_config("training_config.yaml")
cfg.model.moe.backend = "grouped_mm" if probe_grouped_mm() else "fallback"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("tokenizer")
model = Transformer(cfg.model).to(device)
model.load_state_dict(load_file("model.safetensors", device=device))
model.eval()

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 3,908 kB of archives.
After this operation, 11.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 git-lfs amd64 3.4.1-1ubuntu0.4 [3,908 kB]
Fetched 3,908 kB in 0s (13.0 MB/s)
Selecting previously unselected package git-lfs.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...


Transformer(
  (embed): Embedding(49280, 1024)
  (blocks): ModuleList(
    (0-15): 16 x TransformerBlock(
      (norm1): ZeroCenteredRMSNorm()
      (attn): Attention(
        (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (k_proj): Linear(in_features=1024, out_features=256, bias=False)
        (v_proj): Linear(in_features=1024, out_features=256, bias=False)
        (o_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (q_norm): ZeroCenteredRMSNorm()
        (k_norm): ZeroCenteredRMSNorm()
        (gate_proj): Linear(in_features=1024, out_features=1024, bias=False)
      )
      (norm2): ZeroCenteredRMSNorm()
      (moe): MoEFFN(
        (router): Linear(in_features=1024, out_features=64, bias=False)
      )
    )
  )
  (final_norm): ZeroCenteredRMSNorm()
)

### Generate Text

Now, let's write a simple helper function to generate responses from a prompt.

In [3]:
import torch.nn.functional as F

def generate_response(prompt, max_new_tokens=100, temperature=0.8, top_k=50):
    real_vocab_size = len(tokenizer)
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    input_ids = torch.tensor([ids], device=device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            with torch.autocast(device_type=device, dtype=torch.bfloat16 if device == "cuda" else torch.float32):
                logits, _, _, _ = model(input_ids)
            next_logits = logits[0, -1, :real_vocab_size] / temperature
            top_vals, top_idx = next_logits.topk(top_k)
            probs = F.softmax(top_vals.float(), dim=-1)
            next_id = top_idx[torch.multinomial(probs, 1)]
            input_ids = torch.cat([input_ids, next_id.view(1, 1)], dim=1)
            if next_id.item() == tokenizer.eos_token_id:
                break
    return tokenizer.decode(input_ids[0].tolist())

prompt = "Albert Einstein what is his most notable work"
print(f"Prompt: {prompt}\n")
print(f"Response: {generate_response(prompt)}")

Prompt: Albert Einstein what is his most notable work

Response: Albert Einstein what is his most notable work? (36)
Einstein’s most famous work is the general theory of relativity. As the title suggest, the theory is one of the most fundamental laws of the universe. Specifically, it tells us that the speed of light is directly related to the size of the universe. When two bodies are moving away from each other, the larger the separation will be between them. And it is this relationship that allows us to determine the speed of our space ship.
As per Einstein’s


In [ ]:
print("Chat with the model (type 'exit' to stop). Note: this is a BASE model, not instruction-tuned —")
print("it continues text rather than answering like an assistant, so short factual prompts work best.\n")

while True:
    prompt = input("You: ")
    if prompt.strip().lower() in ("exit", "quit"):
        break
    response = generate_response(prompt, max_new_tokens=100)
    print(f"Model: {response}\n")

Chat with the model (type 'exit' to stop). Note: this is a BASE model, not instruction-tuned —
it continues text rather than answering like an assistant, so short factual prompts work best.

You: tell me more about william shakespear
Model: tell me more about william shakespear's the tragedy of othello, shakespeare shows us that the characters of his most well known tragedy, "the tragedy of othello" can be seen as being similar to his character, by the fact that he uses dialogue and description in his speeches.
Shakespeare's othello essays: over 180,000 othello essays, othello term papers, othello research paper, book reports 184 990 essays, term and research papers available for unlimited

You: what is context size in an llm
Model: what is context size in an llm-calculator?
- How do you find the context size in a llm-calculator?
- What is the default context size in an llm-calculator?
- What is the default context size for 3D object?
- What is the default context size in a 3D object?
